# 00 · Environment & data setup check

A quick sanity check before any real work. This notebook confirms that:

1. The `medimg` environment has the libraries we rely on, at the versions we expect.
2. The two raw BraTS datasets are present on disk with the right number of cases.
3. The pre-computed split index (`splits/index.csv`) loads and has the expected per-split counts.

If every cell below runs without an assertion error, the project is ready to go.

## 1. Library versions

Print the Python interpreter and the versions of the core scientific / medical-imaging stack. These are the libraries the rest of the project imports.

In [1]:
import sys
import torch
import monai
import nibabel
import numpy as np
import matplotlib

print(f"Python     {sys.version.split()[0]}")
print(f"torch      {torch.__version__}")
print(f"monai      {monai.__version__}")
print(f"nibabel    {nibabel.__version__}")
print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")

Python     3.11.17
torch      2.14.1+cpu
monai      1.6.1
nibabel    5.4.2
numpy      2.4.6
matplotlib 3.11.2


## 2. Data directories

The raw data lives outside the repo under `C:\data\brats`. We expect two sources:

| Source | Folder | Expected cases |
|--------|--------|----------------|
| GLI (adult glioma, BraTS 2023) | `ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData` | 1251 |
| SSA (BraTS-Africa) | `BraTS-Africa/95_Glioma` | 95 |

We check each directory exists and count the case sub-folders.

In [2]:
from pathlib import Path

DATA = Path(r"C:\data\brats")
SOURCES = {
    "GLI": DATA / "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "SSA": DATA / "BraTS-Africa" / "95_Glioma",
}
EXPECTED = {"GLI": 1251, "SSA": 95}

for name, root in SOURCES.items():
    assert root.exists(), f"Missing data directory: {root}"
    n_cases = sum(1 for p in root.iterdir() if p.is_dir())
    status = "OK" if n_cases == EXPECTED[name] else "MISMATCH"
    print(f"{name}: {n_cases:>4} case folders (expected {EXPECTED[name]})  [{status}]")
    print(f"     {root}")
    assert n_cases == EXPECTED[name], f"{name}: expected {EXPECTED[name]} cases, found {n_cases}"

GLI: 1251 case folders (expected 1251)  [OK]
     C:\data\brats\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
SSA:   95 case folders (expected 95)  [OK]
     C:\data\brats\BraTS-Africa\95_Glioma


## 3. Split index

`splits/index.csv` (built by `scripts/make_index.py`) lists every case, the relative path to each modality, and the split it belongs to. GLI patients are split train/val/test at the **patient** level; the whole of BraTS-Africa is held out as an `external_test` set.

Expected counts: **train 869 / val 192 / test 190 / external_test 95**.

In [3]:
import pandas as pd

# Resolve the repo root whether this notebook runs from notebooks/ or the repo root.
CWD = Path.cwd()
REPO = CWD if (CWD / "splits" / "index.csv").exists() else CWD.parent
index_path = REPO / "splits" / "index.csv"

df = pd.read_csv(index_path)
print(f"Loaded {index_path}  ->  {len(df)} rows, {df.shape[1]} columns")
print()

EXPECTED_SPLITS = {"train": 869, "val": 192, "test": 190, "external_test": 95}
counts = df["split"].value_counts()
for split, expected in EXPECTED_SPLITS.items():
    got = int(counts.get(split, 0))
    status = "OK" if got == expected else "MISMATCH"
    print(f"{split:<14} {got:>4}  (expected {expected})  [{status}]")

for split, expected in EXPECTED_SPLITS.items():
    assert int(counts.get(split, 0)) == expected, f"{split}: expected {expected}"

print(f"\nTotal: {len(df)} cases")

Loaded C:\Users\Rajesh Sampathkumar\Documents\GitHub\brats-robustness\splits\index.csv  ->  1346 rows, 10 columns

train           869  (expected 869)  [OK]
val             192  (expected 192)  [OK]
test            190  (expected 190)  [OK]
external_test    95  (expected 95)  [OK]

Total: 1346 cases


All checks passed — the environment, data, and splits are ready.